In [ ]:
import pymysql
import pandas as pd
import json
import os
import sys
import requests
sys.path.append("/home/trade_core")
from postgres_client import InitDBClient
from psycopg2 import extras
import time

In [ ]:
with open('/home/trade_core/trade_core_config.json') as config_file:
    config = json.load(config_file)

node_db_name = config['node_settings'][config['node']]['db_settings']
node_db_setting_dict = config['database_setting'][node_db_name]

In [ ]:
db_client = InitDBClient(**{**node_db_setting_dict, 'database': 'trade_core'})

In [ ]:
db_client.create_all_tables()

In [ ]:
def is_table_empty(conn, table_name):
    with conn.cursor() as cur:
        cur.execute(f"SELECT COUNT(*) FROM {table_name}")
        count = cur.fetchone()[0]
        return count == 0

def get_column_names(conn, table_name):
    with conn.cursor() as cur:
        cur.execute("""
            SELECT column_name 
            FROM information_schema.columns 
            WHERE table_schema = 'public' AND table_name = %s
            ORDER BY ordinal_position;
        """, (table_name,))
        return [row[0] for row in cur.fetchall()]

In [ ]:
table_name = 'order_history'
conn = db_client.pool.getconn()
curr = conn.cursor(cursor_factory=extras.RealDictCursor)

sql = f"""SELECT * FROM {table_name}"""
curr.execute(sql)
order_history_df = pd.DataFrame(curr.fetchall())
db_client.pool.putconn(conn)


In [ ]:
order_history_df[order_history_df['order_id']=='54092062893']

In [ ]:
conn = db_client.pool.getconn()
curr = conn.cursor(cursor_factory=extras.RealDictCursor)

trade_info_dict = {
    "trade_config_uuid": "e16b3bdc-301b-4512-be3b-f0e60cef7004",
    "trade_uuid": "700469e0-3bea-426e-ae3b-318f1a8c1de7",
    "trade_side": "ENTER",
    "base_asset": "TEST",
    "registered_datetime": None,
    "target_premium_value": 0,
    "executed_premium_value": 0,
    "slippage_p": 0,
    "dollar": 0,
    "remark": None,
    "target_order_id": "45347e3e-f244-4294-8c0b-6d2c57191202",
    "origin_order_id": "54063132846"
}

sql = """INSERT INTO trade_history (trade_config_uuid, trade_uuid, registered_datetime, trade_side, base_asset, target_order_id, origin_order_id, 
            target_premium_value, executed_premium_value, slippage_p, dollar, remark) 
            VALUES (%(trade_config_uuid)s, %(trade_uuid)s, %(registered_datetime)s, %(trade_side)s, %(base_asset)s, 
            %(target_order_id)s, %(origin_order_id)s, %(target_premium_value)s, %(executed_premium_value)s, %(slippage_p)s, %(dollar)s, %(remark)s) RETURNING uuid"""
res = curr.execute(sql, trade_info_dict)
generated_uuid = curr.fetchone()

db_client.pool.putconn(conn)

In [ ]:
generated_uuid['uuid']

In [ ]:
conn = db_client.pool.getconn()
curr = conn.cursor(cursor_factory=extras.RealDictCursor)


A = 'trade'
B = "order_history"
C = "trade_history"


sql = f"""
    SELECT 
    {A}.*,
    {B}.order_id AS b_order_id, {B}.order_type AS b_order_type, {B}.market_code AS b_market_code, 
    {B}.symbol AS b_symbol, {B}.quote_asset AS b_quote_asset, {B}.side AS b_side, 
    {B}.price AS b_price, {B}.qty AS b_qty, {B}.fee AS b_fee, {B}.remark AS b_remark,
    {C}.uuid AS c_uuid, {C}.registered_datetime AS c_registered_datetime, {C}.trade_side AS c_trade_side, 
    {C}.base_asset AS c_base_asset, {C}.target_order_id AS c_target_order_id, 
    {C}.origin_order_id AS c_origin_order_id, {C}.dollar AS c_dollar, {C}.remark AS c_remark
    FROM 
        {A}
        INNER JOIN {B} ON {A}.trade_config_uuid = {B}.trade_config_uuid AND {A}.uuid = {B}.trade_uuid
        INNER JOIN {C} ON {B}.trade_config_uuid = {C}.trade_config_uuid AND {B}.trade_uuid = {C}.trade_uuid

"""
curr = conn.cursor(cursor_factory=extras.RealDictCursor)
curr.execute(sql)
result = curr.fetchall()

db_client.pool.putconn(conn)

In [ ]:
start = time.time()
for i in range(1000):
    conn = db_client.pool.getconn()
    curr = conn.cursor(cursor_factory=extras.RealDictCursor)
    sql = """
        SELECT * FROM trade"""
    curr.execute(sql)
    result = curr.fetchall()
    # print(result)
    db_client.pool.putconn(conn)
print(f"Time: {time.time() - start}")

In [ ]:
start = time.time()
conn = db_client.pool.getconn()
for i in range(1000):
    curr = conn.cursor(cursor_factory=extras.RealDictCursor)
    sql = """
        SELECT * FROM trade"""
    curr.execute(sql)
    result = curr.fetchall()
    # print(result)
db_client.pool.putconn(conn)
print(f"Time: {time.time() - start}")

In [ ]:
start = time.time()
for i in range(1000):
    conn = db_client.pool.getconn()
    curr = conn.cursor(cursor_factory=extras.RealDictCursor)
    sql = """
        SELECT trade.*, trade_config.uuid, trade_config, trade_config.service_datetime_end, trade_config.telegram_id, trade_config.target_market_code, trade_config.origin_market_code
        FROM trade
        JOIN trade_config ON trade.trade_config_uuid = trade_config.uuid WHERE trade_config.target_market_code='UPBIT_SPOT/KRW' AND trade_config.origin_market_code='BINANCE_USD_M/USDT' AND trade_config.service_datetime_end <= %s"""
    val = (pd.Timestamp.now(),)
    curr.execute(sql, val)
    result = curr.fetchall()
    # print(result)
    db_client.pool.putconn(conn)
print(f"Time: {time.time() - start}")

In [ ]:
start = time.time()
conn = db_client.pool.getconn()
for i in range(1000):
    curr = conn.cursor(cursor_factory=extras.RealDictCursor)
    sql = """
        SELECT trade.*, trade_config.uuid, trade_config, trade_config.service_datetime_end, trade_config.telegram_id, trade_config.target_market_code, trade_config.origin_market_code
        FROM trade
        JOIN trade_config ON trade.trade_config_uuid = trade_config.uuid WHERE trade_config.target_market_code='UPBIT_SPOT/KRW' AND trade_config.origin_market_code='BINANCE_USD_M/USDT' AND trade_config.service_datetime_end <= %s"""
    val = (pd.Timestamp.now(),)
    curr.execute(sql, val)
    result = curr.fetchall()
    # print(result)
db_client.pool.putconn(conn)
print(f"Time: {time.time() - start}")

In [ ]:
start = time.time()
conn = db_client.pool.getconn()
for i in range(100):
    curr = conn.cursor(cursor_factory=extras.RealDictCursor)
    sql = """
        SELECT trade.*, trade_config.uuid, trade_config, trade_config.service_datetime_end, trade_config.telegram_id, trade_config.target_market_code, trade_config.origin_market_code
        FROM trade
        JOIN trade_config ON trade.trade_config_uuid = trade_config.uuid"""
    val = (pd.Timestamp.now(),)
    curr.execute(sql)
    result = curr.fetchall()
    # print(result)
db_client.pool.putconn(conn)
print(f"Time: {time.time() - start}")

In [ ]:
result

In [ ]:
0.36/1000

In [ ]:
# First check whether the table is empty
conn = db_client.pool.getconn()
curr = conn.cursor(cursor_factory=extras.RealDictCursor)
if is_table_empty(conn, 'trade_config'):
    # # Get the column names
    # column_names = self.get_column_names(conn, table_name)
    # # Create empty dataframe
    # self.exchange_config_df = pd.DataFrame(columns=column_names)
    pass
else:
    curr.execute(f"SELECT * FROM {table_name}")
    exchange_config_df = pd.DataFrame(curr.fetchall())
    target_market_code_unique = exchange_config_df['target_market_code'].unique()
    origin_market_code_unique = exchange_config_df['origin_market_code'].unique()
    for target_market_code in target_market_code_unique:
        for origin_market_code in origin_market_code_unique:
            market_combi_code = f"{target_market_code}:{origin_market_code}"
            self.exchange_config_df_dict[market_combi_code] = exchange_config_df[(exchange_config_df['target_market_code'] == target_market_code) & (exchange_config_df['origin_market_code'] == origin_market_code)]
self.postgres_client.pool.putconn(conn)
if self.exchange_config_df_dict_initiated is False:
    self.exchange_config_df_dict_initiated = True

In [ ]:
response = requests.get("https://arbicrypto.net/api/users/users/")